# Pinecone RAG Training Pipeline

Fine-tune an LLM on your documents using reinforcement learning with Pinecone as the retrieval backend. This notebook chunks your documents, generates synthetic QA pairs, and launches a training job -- edit the configuration cell below and run top-to-bottom.

In [ ]:
# Uncomment for Google Colab
# !pip install cgft[pinecone] openai

In [ ]:
%load_ext autoreload
%autoreload 2

from cgft.utils import ensure_safe_python_version

ensure_safe_python_version()

## Configuration

In [ ]:
# Pinecone credentials -- https://app.pinecone.io/
PINECONE_API_KEY = ""
INDEX_NAME = "my-index"
INDEX_HOST = ""   # Leave empty to auto-resolve from index name
NAMESPACE = ""    # Leave empty for default namespace
EMBED_MODEL = "multilingual-e5-large"

# CGFT platform credentials -- https://app.cgft.io/account/api-keys
API_KEY = ""
BASE_URL = "https://app.cgft.io"

# LLM credentials -- used for QA generation, filtering, and refinement
LLM_API_KEY = ""
LLM_BASE_URL = ""
LLM_MODEL = "grok-4-fast-non-reasoning"

# Judge LLM credentials -- used at reward time to evaluate correctness + citation
JUDGE_API_KEY = LLM_API_KEY
JUDGE_BASE_URL = LLM_BASE_URL
JUDGE_MODEL = "grok-4-fast-non-reasoning"

# Documents to train on (local directory of markdown/text files)
DOCS_PATH = "./samples/posthog/docs/"

# Corpus context -- helps the QA generator understand your documents
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation
TOTAL_SAMPLES = 100
OUTPUT_DIR = "outputs/pinecone_rag"

# Training experiment
EXPERIMENT_NAME = "my-search-v1"
EXPERIMENT_PREFIX = "pinecone-search"

## Step 1: Chunk & Upload Documents

In [4]:
from cgft.corpus.pinecone.source import PineconeChunkSource

source = PineconeChunkSource(
    api_key=PINECONE_API_KEY,
    index_name=INDEX_NAME,
    index_host=INDEX_HOST or None,
    namespace=NAMESPACE,
    embed_model=EMBED_MODEL,
)
source.populate_from_folder(DOCS_PATH)

Chunking documents from ./samples/posthog/docs/...
ChunkCollection Summary
Total chunks: 3498
Total files: 1264

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1150 chars

File Structure:
├── data-warehouse/
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── reddit-ads.mdx (1 chunks)
│   │   ├── s3.mdx (1 chunks)
│   │   ├── temporal.mdx (1 chunks)
│   │   ├── klaviyo.mdx (1 chunks)
│   │       ... 36 more files
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── sql/
│   │   ├── variables.mdx (2 chunks)
│   │   ├── index.mdx (8 chunks)
│   │   ├── data-access.mdx (3 chunks)
│   │   └── useful-functions.mdx (7 chunks)
│   ├── under-the-hood.md (1 chunks)
│   ├── cutting-costs.mdx (3 chunks)
│   ├── join.mdx (2 chunks)
│   ├── run-sql-mcp.mdx (4 chunks)
│       ... 8 more files
├── advanced/
│   ├── proxy/
│   │   ├── 

PineconeConfigurationError: You haven't specified an API key. Please either set the PINECONE_API_KEY environment variable or pass the 'api_key' keyword argument to the Pinecone client constructor.

## Step 2: Generate Synthetic QA

In [ ]:
from cgft.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LLMDirectGenerationConfig,
    OutputConfig,
    PlatformConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
)
from cgft.qa_generation.cgft_pipeline import CgftPipeline

cfg = CgftPipelineConfig(
    platform=PlatformConfig(
        api_key=API_KEY,
        base_url=BASE_URL,
        llm_api_key=LLM_API_KEY,
        llm_base_url=LLM_BASE_URL,
    ),
    corpus=CorpusConfig(corpus_name="rag-corpus", min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
    ),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(model=LLM_MODEL),
    ),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(judge_model=LLM_MODEL),
        grounding_llm=GroundingLLMFilterConfig(judge_model=LLM_MODEL),
    ),
    targets=TargetsConfig(total_samples=TOTAL_SAMPLES),
    output=OutputConfig(dir=OUTPUT_DIR),
)
cfg.resolve_api_keys()

In [ ]:
pipeline = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = pipeline.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]

## Step 3: Launch Training

In [ ]:
import cgft
from cgft.corpus.pinecone.search import PineconeSearch
from cgft.envs.search_env import SearchEnv
from cgft.trainer.pipeline import train

search = PineconeSearch(
    api_key=PINECONE_API_KEY,
    index_name=INDEX_NAME,
    index_host=INDEX_HOST or None,
    namespace=NAMESPACE,
    embed_model=EMBED_MODEL,
)

experiment_id = train(
    env_class=SearchEnv,
    env_args={
        "search": search,
        "judge_base_url": JUDGE_BASE_URL,
        "judge_api_key": JUDGE_API_KEY,
        "judge_model": JUDGE_MODEL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix=EXPERIMENT_PREFIX,
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[cgft],
    pip_dependencies=["pinecone", "openai"],
    experiment_name=EXPERIMENT_NAME,
    validate_env_remotely=True,
)

## What to Expect

Training runs take 30-60 minutes. Monitor at [app.cgft.io](https://app.cgft.io).

**Early training**: Rewards will fluctuate and metrics may be noisy. This is normal -- the model is learning basic patterns.

**Healthy trajectory**: pass@k increases first (model solves more prompts), then average reward follows (model gets better at each prompt).

**Warning signs**:
- pass@k flat while reward increases -- overfitting to easy prompts
- Both stagnate early -- training distribution too hard or reward signal too sparse